In [ ]:
import os
from pathlib import Path
# Configurable base directory for data and research files.

# Set GLOBROT_DATA environment variable to override the default.

BASE_DATA_DIR = Path(os.environ.get('GLOBROT_DATA', './Research/LangevinDynamics_RotationalDiffusion')).resolve()
# Common derived locations used in the notebooks (fall back to BASE_DATA_DIR-related defaults)

CorrFuncLOC = str(Path(os.environ.get('GLOBROT_DOCS', './Documents/Research/DiffusionTip4pDSoluteSize')).resolve()) + '/'
CorrFuncLoc2 = str(BASE_DATA_DIR) + '/'


def _remap_drive_path(s):

    """If s is a Windows absolute path (drive letter), remap it under BASE_DATA_DIR by stripping the drive."""

    if not isinstance(s, str):

        return s

    m = DRIVE_RE.match(s)

    if not m:

        return s

    tail = DRIVE_RE.sub('', s)

    # Normalize separators and join to base

    tail = tail.replace('\\', '/').lstrip('/')

    return str(BASE_DATA_DIR.joinpath(tail))


In [ ]:
import tarfile
import codecs
import io
import os 
import sys

import numpy as np
import pandas as pd
import mdtraj as md
from scipy.optimize import curve_fit
from scipy.integrate import simps

import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib import ticker as mticker

In [ ]:
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'sans:italic:bold'
plt.rcParams['mathtext.bf'] = 'sans:bold'

In [ ]:
def calc_chi(y1, y2, dy=[]):
    if dy != []:
        return np.sum( (y1-y2)**2.0/dy )/len(y1)
    else:
        return np.sum( (y1-y2)**2.0 )/len(y1)

In [ ]:
def _bound_check(func, params):
    """
    Hack for now.
    """
    if len(params) == 1:
        return False
    elif len(params)%2 == 0 :
        s = sum(params[0::2])
        return (s>1)
    else:
        s = params[0]+sum(params[1::2])
        return (s>1)

In [ ]:
## Functions 1,3,5,7,9 are the functions that the sum of coefficients are equal to 1. They have one less parameter.
## Functions 2,4,6,8,10 are the functions where the sum of coefficients are not restricted.

def func_exp_decay1(t, tau_a):
    return np.exp(-t/tau_a)
def func_exp_decay2(t, A, tau_a):
    return A*np.exp(-t/tau_a)
def func_exp_decay3(t, A, tau_a, tau_b):
    return A*np.exp(-t/tau_a) + (1-A)*np.exp(-t/tau_b)
def func_exp_decay4(t, A, tau_a, B, tau_b ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b)
def func_exp_decay5(t, A, tau_a, B, tau_b, tau_g ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + (1-A-B)*np.exp(-t/tau_g)
def func_exp_decay6(t, A, tau_a, B, tau_b, G, tau_g ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g)
def func_exp_decay7(t, A, tau_a, B, tau_b, G, tau_g, tau_d):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + (1-A-B-G)*np.exp(-t/tau_d)
def func_exp_decay8(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d)
def func_exp_decay9(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d, tau_e):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d) + (1-A-B-G-D)*np.exp(-t/tau_e)
def func_exp_decay10(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d, E, tau_e):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d) + E*np.exp(-t/tau_e)


In [ ]:
def _return_parameter_names(num_pars):
    if num_pars==1:
        return ['C_a', 'tau_a']
    elif num_pars==2:
         return ['C_a', 'tau_a']
    elif num_pars==3:
         return ['C_a', 'tau_a', 'tau_b']
    elif num_pars==4:
         return ['C_a', 'tau_a', 'C_b', 'tau_b']
    elif num_pars==5:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'tau_g']
    elif num_pars==6:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g']
    elif num_pars==7:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'tau_d']
    elif num_pars==8:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d']
    elif num_pars==9:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d', 'tau_e']
    elif num_pars==10:
         return [ 'C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d', 'C_e', 'tau_e']

    return []

In [ ]:
def do_Expstyle_fit2(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005):
    
    """
    Function that calculates the fit to the exponential function
    Chooses the function based off the number of parameters
    tmin is the minimum value the correlation times can take;
    usually set to the timestep used to calculate the correlation function; in units of the timestep;
    tinput: the time value to scale the initial guesses.
    tmax: maximum bounding value of the correlation times
    tmin: minimum bounding value of the correlation times: usually x[t=1]
    """
    ## These are really bad guesses 
    b1_guess = [1 - y[0]*0.99,  y[0]/3+0.15, y[0]/2 +0.3, y[0]] 
    #b1_guess = 1.0/num_pars/2
    t1_guess = [tinput/1280, tinput/640, tinput/16, tinput/4]
    
    tmax=tmax*1.5
    tmax_values = [0.0099, 0.350, 3.5, tmax]
    
    if num_pars==1:
        func=func_exp_decay1
        guess=(t1_guess[2])
        bound=(0.,tmax)
    elif num_pars==2:
        func=func_exp_decay2
        guess=(b1_guess[3], t1_guess[2])
        bound=([0.0, tmin], [1., tmax])
    elif num_pars==3:
        func=func_exp_decay3
        guess=(b1_guess[3], t1_guess[3], t1_guess[0])
        bound=([0.0, tmin, tmin],[1., tmax, tmax])
    elif num_pars==4:
        func=func_exp_decay4
        guess = (b1_guess[3], t1_guess[3], b1_guess[0], t1_guess[1])
        #guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[1])
        bound=([0.0, tmin, 0.0, tmin], [1.0, tmax, 1.0, tmax_values[1]])
    elif num_pars==5:
        func=func_exp_decay5
        guess=(b1_guess[3], t1_guess[3], b1_guess[1], t1_guess[2], t1_guess[1])
        bound=([0.0, tmin, 0.0, tmin, tmin],[1., tmax, 1., tmax, tmax])
    elif num_pars==6:
        func=func_exp_decay6
        guess=(b1_guess[3], t1_guess[3], b1_guess[2], t1_guess[2], b1_guess[0], t1_guess[0])
        bound=([0.0, x[1], 0.0, x[1], 0.0, tmin],[1., tmax, 1., tmax, 1., tmax])
    elif num_pars==7:
        func=func_exp_decay7
        guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[2], b1_guess, t1_guess[1],
               t1_guess[0])
        bound=([0.0, x[0], 0.0, x[0], 0.0, x[0], tmin],[1., tmax, 1., tmax, 1., tmax, tmax])
    elif num_pars==8:
        func=func_exp_decay8
        guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[2], b1_guess, t1_guess[1],
               b1_guess, t1_guess[0])
        bound=([0.0, x[0], 0.0, x[0], 0.0, x[0], 0.0, tmin],[1., tmax, 1., tmax, 1., tmax, 1., tmax])

    if len(dy) > 1:
        popt, popv = curve_fit(func, x, y, p0=guess, sigma=dy, bounds=bound, method='trf', loss='linear', 
                               max_nfev=num_pars*1000)
    else:
        popt, popv = curve_fit(func, x, y, p0=guess, bounds=bound) ##, loss='soft_l1')

    ymodel=[ func(x[i], *popt) for i in range(len(x)) ]
    #print ymodel

    bExceed=_bound_check(func, popt)
    if bExceed:
        print( "= = = WARNING, curve fitting in do_LSstyle_fit returns a sum>1.//")
        return 9999.99, popt, popv, ymodel
    else:
        return calc_chi(y, ymodel, dy), popt, popv, ymodel

In [ ]:
def read_CorrFuncTarball(atom_df, prot, ens, dt, dtau, gamma = 'Coupling2ps', corr_type='NH'):
    
    corr_time_index = np.arange(0, dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index=corr_time_index, columns=atom_df['RES_ATM_Pair'].values)
    rescorrdf_STD = rescorrdf.copy()
    
    tarfname = '{}{}/METHYL_C13HCorrFuncData.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
    with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
        
        for mindx, methylc in atom_df.iterrows():
            print(methylc.ATMID)
            rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
            
            for rind in RUNS:
                
                hydcorrF = pd.DataFrame(index=corr_time_index, columns=['H1','H2','H3'])
                
                for hyd in ['H1','H2','H3']:
                    
                    corrfuncname = '13C{}_{}{}CorrelationFunctions.dat'.format(methylc.ATMID, hyd, methylc.RESID)
                    if ens == 'NVE':
                        tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
                    elif ens =='NVE2':
                        tarmemberloc = './PROD_NVE_Run2/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
                    elif ens =='NVT':
                        tarmemberloc = './PROD_NVT/{}/Analysis/{}/METHYLCorrFunc/{}'.format(gamma,rind, corrfuncname)
                    print(tarmemberloc)    
                    cffile_extract = corrtar.extractfile(tarmemberloc).read()
                    CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
                    hydcorrF.loc[:, hyd] = CorrFuncDF['<P2>'].values
                
                rescfdf.loc[:, rind] = hydcorrF.mean(axis=1)
            
            rescorrdf[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
            rescorrdf_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
            zero_std = rescorrdf_STD[methylc.RES_ATM_Pair][rescorrdf_STD[methylc.RES_ATM_Pair]==0.0].index
            rescorrdf_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6
    
    return rescorrdf, rescorrdf_STD

In [ ]:
def read_CorrFuncTarball_C13C13(atom_df, prot, ens, dt, dtau, gamma = 'Coupling2ps', corr_type='NH'):
    
    corr_time_index = np.arange(0, dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index=corr_time_index, columns=atom_df['RES_ATM_Pair'].values)
    rescorrdf_STD = rescorrdf.copy()
    
    tarfname = '{}{}/UBQ_MethylC13C13CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
    with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
        
        for mindx, methylc in atom_df.iterrows():
            print(methylc.ATMID)
            rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
            corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
            for rind in RUNS:
                
                
                if ens == 'NVE':
                    tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
                elif ens =='NVE2':
                    tarmemberloc = './PROD_NVE_Run2/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
                elif ens =='NVT':
                    tarmemberloc = './PROD_NVT/{}/Analysis/{}/METHYLCorrFunc/{}'.format(gamma,rind, corrfuncname)
                print(tarmemberloc)    
                
                cffile_extract = corrtar.extractfile(tarmemberloc).read()
                CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
                rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
            rescorrdf[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
            rescorrdf_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
            zero_std = rescorrdf_STD[methylc.RES_ATM_Pair][rescorrdf_STD[methylc.RES_ATM_Pair]==0.0].index
            rescorrdf_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6
    
    return rescorrdf, rescorrdf_STD

In [ ]:
def read_CorrFuncTarball_OptNVE(atom_df, opt_simlist, prot, dt, dtau, corr_type='NH'):
    
    """ Only for NVE ensemble !!! """
    
    corr_time_index = np.arange(0, dtau+1,  1)*dt/1e3
    rescorrdf = pd.DataFrame(index=corr_time_index, columns=atom_df['RES_ATM_Pair'].values)
    rescorrdf_STD = rescorrdf.copy()
    
    tarfname = '{}{}/METHYL_C13HCorrFuncData.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
    with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
        
        for mindx, methylc in atom_df.iterrows():
            print(methylc.ATMID)
            rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
            
            for no, opts in enumerate(opt_simlist):
                rind = 'Run{}'.format(no+1)
                hydcorrF = pd.DataFrame(index=corr_time_index, columns=['H1','H2','H3'])
                
                for hyd in ['H1','H2','H3']:
                    
                    corrfuncname = '13C{}_{}{}CorrelationFunctions.dat'.format(methylc.ATMID, hyd, methylc.RESID)
                    tarmemberloc = './{}/Analysis/{}/METHYLCorrFunc/{}'.format(opts[0], opts[1], corrfuncname)
                        
                    print(tarmemberloc)    
                    cffile_extract = corrtar.extractfile(tarmemberloc).read()
                    CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
                    hydcorrF.loc[:, hyd] = CorrFuncDF['<P2>'].values
                
                rescfdf.loc[:, rind] = hydcorrF.mean(axis=1)
            
            rescorrdf[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
            rescorrdf_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
            zero_std = rescorrdf_STD[methylc.RES_ATM_Pair][rescorrdf_STD[methylc.RES_ATM_Pair]==0.0].index
            rescorrdf_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6
    
    return rescorrdf, rescorrdf_STD

In [ ]:
def read_CorrFuncTarball_C13C13_OptNVE(atom_df, opt_simlist, prot, ens, dt, dtau, corr_type='NH'):
    
    corr_time_index = np.arange(0, dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index=corr_time_index, columns=atom_df['RES_ATM_Pair'].values)
    rescorrdf_STD = rescorrdf.copy()
    
    tarfname = '{}{}/UBQ_MethylC13C13CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
    with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
        
        for mindx, methylc in atom_df.iterrows():
            print(methylc.ATMID)
            rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
            corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
            
            for no, opts in enumerate(opt_simlist):
                rind = 'Run{}'.format(no+1)
                    
                tarmemberloc = './{}/Analysis/{}/METHYLCorrFunc/{}'.format(opts[0], opts[1], corrfuncname)

                print(tarmemberloc)    
                
                cffile_extract = corrtar.extractfile(tarmemberloc).read()
                CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
                rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
            rescorrdf[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
            rescorrdf_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
            zero_std = rescorrdf_STD[methylc.RES_ATM_Pair][rescorrdf_STD[methylc.RES_ATM_Pair]==0.0].index
            rescorrdf_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6
    
    return rescorrdf, rescorrdf_STD

In [ ]:
def _read_CorrFunctions(atom_df, opt_simlist, prot, ens, dt, dtau, corr_type='NH', gamma = 'CF2ps'):
    
    """
    Function to read the correlation functions calculated from cpptraj 
    
    """
    corr_time_index = np.arange(0, dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index = corr_time_index, columns=atom_df.iloc[1:]['RESID'])
    rescorrdf_STD = rescorrdf.copy()
    for rcol in rescorrdf.columns:
        rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
        for R in RUNS:
            
            if ens == 'NVE':
                corrdf = pd.read_csv('{}/{}CorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF_NVE.loc[R,(prot,ens)],
                                                                                        corr_type, rcol),
                                    delim_whitespace=True)
            elif ens =='NVT':
                corrdf = pd.read_csv('{}/{}CorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF_NVT.loc[R,(prot,gamma)],
                                                                                        corr_type, rcol),
                                    delim_whitespace=True)
                
            rescfdf.loc[:,R] = corrdf['<P2>'].values
            
        rescorrdf[rcol] = rescfdf.mean(axis=1)
        rescorrdf_STD[rcol] = rescfdf.std(axis=1)
        
        zero_std = rescorrdf_STD[rcol][rescorrdf_STD[rcol]==0.0].index
        rescorrdf_STD.loc[zero_std,rcol] = 1e-6
    
    return rescorrdf, rescorrdf_STD
    

In [ ]:
CorrFuncLOC = './Documents/Research/DiffusionTip4pDSoluteSize/'
CorrFuncLoc2 = './Research/LangevinDynamics_RotationalDiffusion/'
ENSEMBLES = ['NVE', 'NVT']
CouplingFrequency = ['CF0-2ps', 'CF2ps', 'CF20ps']
RUNS = ['Run{}'.format(n) for n in range(1,5,1)]
Proteins = ['GB3', 'Ubiquitin']
MINDX = pd.MultiIndex.from_product([Proteins, ENSEMBLES])
MINDX_NVT=pd.MultiIndex.from_product([Proteins, CouplingFrequency])
LocDF_NVE = pd.DataFrame(index=RUNS, columns=MINDX)
LocDF_NVE_R2 = LocDF_NVE.copy()
LocDF_NVE_R3 = LocDF_NVE.copy()
LocDF_NVT = pd.DataFrame(index=RUNS, columns=MINDX_NVT)
FileLocDict = {}

In [ ]:
for items, values in LocDF_NVT.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVT.loc[indx, items] = '{}{}/PROD_NVT/{}/{}'.format(CorrFuncLoc2, items[0], items[1], indx)
        
for items, values in LocDF_NVE.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVE.loc[indx, items] = '{}{}/PROD_{}/{}'.format(CorrFuncLoc2, items[0], items[1], indx)

for items, values in LocDF_NVE_R2.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVE_R2.loc[indx, items] = '{}{}/PROD_{}_Run2/{}'.format(CorrFuncLoc2, items[0], items[1], indx)
        
for items, values in LocDF_NVE_R3.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVE_R3.loc[indx, items] = '{}{}/PROD_{}_Run3/{}'.format(CorrFuncLoc2, items[0], items[1], indx)

In [ ]:
optimalUBQLoc = [('PROD_NVE', 'Run1'), ('PROD_NVE','Run2'), ('PROD_NVE_Run2', 'Run1'), ('PROD_NVE_Run2','Run3')]

In [ ]:
UBQTop = md.load_prmtop('{}Ubiquitin/UBQ_10mMNaCl_FF14SB.prmtop'.format(CorrFuncLoc2))
UBQCAInd = UBQTop.select('name CA and protein')
UBQCAatom_name = pd.Series(['{}'.format(UBQTop.atom(atmx)) for atmx in UBQCAInd])
UBQatom_df = UBQCAatom_name.str.split('-',expand=True).rename(columns={0:'RESNAME',1:'ATMNAME'})
UBQatom_df['RESNAME'] = UBQatom_df['RESNAME'].apply(lambda rr: '{}'.format(rr[:3] + str(int(rr[3:])+1)))
UBQatom_df['RESID'] = UBQatom_df['RESNAME'].apply(lambda rr: int(rr[3:]))

In [ ]:
MethylCarbonDict = {'MET':'CE', 'ALA':'CB', 'THR':'CG2', 'VAL':'CG1 CG2', 'ILE':'CD1 CG2', 'LEU':'CD1 CD2'}
MethylCCDict = {"ALA_CB":"CA", "MET_CE":"SD" ,"LEU_CD1":"CG", "LEU_CD2":"CG",
                "VAL_CG1":"CB","VAL_CG2":"CB", "THR_CG2":"CB", "ILE_CG2":"CB", "ILE_CD1":"CG1"}
MethylCarbonSelect = [UBQTop.select('resname {} and name {}'.format(mckey, MethylCarbonDict[mckey])).flatten() for mckey in MethylCarbonDict.keys()]
MethylCarbonSelect = np.sort(np.hstack(MethylCarbonSelect)).astype(int)
METHYLCarbonName = pd.Series(['{}'.format(UBQTop.atom(atmx)) for atmx in MethylCarbonSelect])
METHYLCarbon_df = METHYLCarbonName.str.split('-',expand=True).rename(columns={0:'RESNAME',1:'ATMNAME'})
METHYLCarbon_df['RESNAME'] = METHYLCarbon_df['RESNAME'].apply(lambda rr: '{}'.format(rr[:3] + str(int(rr[3:])+1)))
METHYLCarbon_df['RES_ATM_Pair'] = METHYLCarbon_df.apply(lambda rr: '{}-{}'.format(rr.RESNAME, rr.ATMNAME),axis=1)
METHYLCarbon_df['RESID'] = METHYLCarbon_df['RESNAME'].apply(lambda rr: int(rr[3:]))
METHYLCarbon_df['ATMID'] = MethylCarbonSelect+1
METHYLCarbon_df['Carbon_Pair'] = METHYLCarbon_df.apply(lambda rr: MethylCCDict['{}_{}'.format(rr.RESNAME[:3],rr.ATMNAME)], axis=1)
#ALACarbonSelect = UBQTop.select('resname ALA and name CB')

In [ ]:
METHYLCarbon_df

In [ ]:
MethylCCDict = {"ALA_CB":"CA", "MET_CE":"SD" ,"LEU_CD1":"CG", "LEU_CD2":"CG",
                "VAL_CG1":"CB","VAL_CG2":"CB", "THR":"CB", "ILE_CG2":"CB", "ILE_CD1":"CG1"}


In [ ]:
UBQ_NVEDF = read_CorrFuncTarball(METHYLCarbon_df, opt_simlist, 'Ubiquitin', 'NVE2', 0.5, 500000, gamma = 'Coupling2ps')

In [ ]:
UBQ_OptNVEDF_C13C13 = read_CorrFuncTarball_C13C13_OptNVE(METHYLCarbon_df.iloc[1:], optimalUBQLoc, 'Ubiquitin', 'NVE', 0.5, 500000)

In [ ]:
corr_time_index = np.arange(0, 500000+1, 1)*0.5/1e3
rescorrdf_MET1_ONVE = pd.DataFrame(index=corr_time_index, columns=METHYLCarbon_df['RES_ATM_Pair'].values)
rescorrdf_MET1_ONVE_STD = rescorrdf_MET1_ONVE.copy()
    
tarfname = '{}{}/MET1_SDC14_CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
    mindx=0
    methylc=METHYLCarbon_df.iloc[0]
    rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
    corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
    
    for no, opts in enumerate(optimalUBQLoc):
        rind = 'Run{}'.format(no+1)
        tarmemberloc = './{}/Analysis/{}/METHYLCorrFunc/{}'.format(opts[0], opts[1], corrfuncname)
        print(tarmemberloc)    
        cffile_extract = corrtar.extractfile(tarmemberloc).read()
        CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
        rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
    rescorrdf_MET1_ONVE[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
    rescorrdf_MET1_ONVE_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
    zero_std = rescorrdf_MET1_ONVE_STD[methylc.RES_ATM_Pair][rescorrdf_MET1_ONVE_STD[methylc.RES_ATM_Pair]==0.0].index
    rescorrdf_MET1_ONVE_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6

In [ ]:
UBQ_OptNVEDF_C13C13_AVE = pd.read_csv('{}{}/PROD_NVE/OptNVE_MethylC13C13_CorrelationFunctions_AVG.csv'.format(CorrFuncLoc2,'Ubiquitin'), index_col=0)
UBQ_OptNVEDF_C13C13_STD = pd.read_csv('{}{}/PROD_NVE/OptNVE_MethylC13C13_CorrelationFunctions_STD.csv'.format(CorrFuncLoc2,'Ubiquitin'), index_col=0)



In [ ]:
rescorrdf_MET1_ONVE[UBQ_OptNVEDF_C13C13[0].columns] = UBQ_OptNVEDF_C13C13[0]
rescorrdf_MET1_ONVE_STD[UBQ_OptNVEDF_C13C13[1].columns] = UBQ_OptNVEDF_C13C13[1]

In [ ]:
rescorrdf_MET1_ONVE.to_csv('{}{}/PROD_NVE/OptNVE_MethylC13C13_CorrelationFunctions_AVG.csv'.format(CorrFuncLoc2,'Ubiquitin'))
rescorrdf_MET1_ONVE_STD.to_csv('{}{}/PROD_NVE/OptNVE_MethylC13C13_CorrelationFunctions_STD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
UBQ_NVEDF_C13C13 = read_CorrFuncTarball_C13C13(METHYLCarbon_df, 'Ubiquitin', 'NVE', 0.5, 500000)

In [ ]:
UBQ_NVEDF_OptSim = read_CorrFuncTarball_OptNVE(METHYLCarbon_df, optimalUBQLoc, 'Ubiquitin', 0.5, 500000)

In [ ]:
AveUBQ_NVT_Cf2ps_C13H

In [ ]:
UBQ_NVEDF_OptSim[0].to_csv('{}{}/PROD_NVE/METHYL_13CH_OptSimCorrFunctionsAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
UBQ_NVEDF_OptSim[1].to_csv('{}{}/PROD_NVE/METHYL_13CH_OptSimCorrFunctionsSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
AveUBQ_NVE_C13H = UBQ_NVEDF_OptSim[0]
StdUBQ_NVE_C13H = UBQ_NVEDF_OptSim[1]

In [ ]:
AveUBQ_NVT_Cf2ps_C13H, StdUBQ_NVT_Cf2ps_C13H = read_CorrFuncTarball(METHYLCarbon_df, 'Ubiquitin', 'NVT', 0.5, 500000, gamma = 'Coupling2ps')

In [ ]:
AveUBQ_NVT_Cf2ps_C13C13, StdUBQ_NVT_Cf2ps_C13C13 = read_CorrFuncTarball_C13C13(METHYLCarbon_df.iloc[1:], 'Ubiquitin', 'NVT', 0.5, 500000, gamma = 'Coupling2ps')

In [ ]:
corr_time_index = np.arange(0, 500000+1, 1)*0.5/1e3
rescorrdf_MET1_NVTCF2ps = pd.DataFrame(index=corr_time_index, columns=METHYLCarbon_df['RES_ATM_Pair'].values)
rescorrdf_MET1_NVTCF2ps_STD = rescorrdf_MET1_NVTCF2ps.copy()
ens='NVT'; gamma='Coupling2ps';
tarfname = '{}{}/MET1_SDC14_CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
    mindx=0
    methylc=METHYLCarbon_df.iloc[0]
    print(methylc.ATMID)
    rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
    corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
    for rind in RUNS:                
                
        if ens == 'NVE':
            tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVE2':
            tarmemberloc = './PROD_NVE_Run2/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVT':
            tarmemberloc = './PROD_NVT/{}/Analysis/{}/METHYLCorrFunc/{}'.format(gamma,rind, corrfuncname)
        print(tarmemberloc)    
        cffile_extract = corrtar.extractfile(tarmemberloc).read()
        CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
        rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
    rescorrdf_MET1_NVTCF2ps[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
    rescorrdf_MET1_NVTCF2ps_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
    zero_std = rescorrdf_MET1_NVTCF2ps_STD[methylc.RES_ATM_Pair][rescorrdf_MET1_NVTCF2ps_STD[methylc.RES_ATM_Pair]==0.0].index
    rescorrdf_MET1_NVTCF2ps_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6

In [ ]:
rescorrdf_MET1_NVTCF2ps[AveUBQ_NVT_Cf2ps_C13C13.columns] = AveUBQ_NVT_Cf2ps_C13C13
rescorrdf_MET1_NVTCF2ps_STD[StdUBQ_NVT_Cf2ps_C13C13.columns] = StdUBQ_NVT_Cf2ps_C13C13

In [ ]:
rescorrdf_MET1_NVTCF2ps.to_csv('{}{}/PROD_NVT/CF2ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
rescorrdf_MET1_NVTCF2ps_STD.to_csv('{}{}/PROD_NVT/CF2ps/METHYL_C13C13_CorrFuncSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
AveUBQ_NVT_Cf02ps_C13C13, StdUBQ_NVT_Cf02ps_C13C13 = read_CorrFuncTarball_C13C13(METHYLCarbon_df.iloc[1:], 'Ubiquitin', 'NVT', 0.5, 500000, gamma = 'Coupling0-2ps')

In [ ]:
corr_time_index = np.arange(0, 500000+1, 1)*0.5/1e3
rescorrdf_MET1_NVTCF02ps = pd.DataFrame(index=corr_time_index, columns=METHYLCarbon_df['RES_ATM_Pair'].values)
rescorrdf_MET1_NVTCF02ps_STD = rescorrdf_MET1_NVTCF02ps.copy()
ens='NVT'; gamma='Coupling0-2ps';
tarfname = '{}{}/MET1_SDC14_CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
    mindx=0
    methylc=METHYLCarbon_df.iloc[0]
    print(methylc.ATMID)
    rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
    corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
    for rind in RUNS:                
                
        if ens == 'NVE':
            tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVE2':
            tarmemberloc = './PROD_NVE_Run2/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVT':
            tarmemberloc = './PROD_NVT/{}/Analysis/{}/METHYLCorrFunc/{}'.format(gamma,rind, corrfuncname)
        print(tarmemberloc)    
        cffile_extract = corrtar.extractfile(tarmemberloc).read()
        CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
        rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
    rescorrdf_MET1_NVTCF02ps[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
    rescorrdf_MET1_NVTCF02ps_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
    zero_std = rescorrdf_MET1_NVTCF02ps_STD[methylc.RES_ATM_Pair][rescorrdf_MET1_NVTCF02ps_STD[methylc.RES_ATM_Pair]==0.0].index
    rescorrdf_MET1_NVTCF02ps_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6

In [ ]:
rescorrdf_MET1_NVTCF02ps[AveUBQ_NVT_Cf02ps_C13C13.columns] = AveUBQ_NVT_Cf02ps_C13C13
rescorrdf_MET1_NVTCF02ps_STD[StdUBQ_NVT_Cf02ps_C13C13.columns] = StdUBQ_NVT_Cf02ps_C13C13
rescorrdf_MET1_NVTCF02ps.to_csv('{}{}/PROD_NVT/CF0-2ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
rescorrdf_MET1_NVTCF02ps_STD.to_csv('{}{}/PROD_NVT/CF0-2ps/METHYL_C13C13_CorrFuncSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
rescorrdf_MET1_NVTCF02ps.to_csv('{}{}/PROD_NVT/CF0-2ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
rescorrdf_MET1_NVTCF02ps_STD.to_csv('{}{}/PROD_NVT/CF0-2ps/METHYL_C13C13_CorrFuncSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
AveUBQ_NVT_Cf20ps_C13C13, StdUBQ_NVT_Cf20ps_C13C13 = read_CorrFuncTarball_C13C13(METHYLCarbon_df.iloc[1:], 'Ubiquitin', 'NVT', 0.5, 500000, gamma = 'Coupling20ps')

In [ ]:
corr_time_index = np.arange(0, 500000+1, 1)*0.5/1e3
rescorrdf_MET1_NVTCF20ps = pd.DataFrame(index=corr_time_index, columns=METHYLCarbon_df['RES_ATM_Pair'].values)
rescorrdf_MET1_NVTCF20ps_STD = rescorrdf_MET1_NVTCF20ps.copy()
ens='NVT'; gamma='Coupling20ps';
tarfname = '{}{}/MET1_SDC14_CorrFunc.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
    mindx=0
    methylc=METHYLCarbon_df.iloc[0]
    print(methylc.ATMID)
    rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
    corrfuncname = '{}_{}C{}CorrFunc.dat'.format(methylc.RESNAME, methylc.Carbon_Pair, methylc.ATMID)
    for rind in RUNS:                
                
        if ens == 'NVE':
            tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVE2':
            tarmemberloc = './PROD_NVE_Run2/Analysis/{}/METHYLCorrFunc/{}'.format(rind, corrfuncname)
                        
        elif ens =='NVT':
            tarmemberloc = './PROD_NVT/{}/Analysis/{}/METHYLCorrFunc/{}'.format(gamma,rind, corrfuncname)
        print(tarmemberloc)    
        cffile_extract = corrtar.extractfile(tarmemberloc).read()
        CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
                    
        rescfdf.loc[:, rind] = CorrFuncDF['<P2>'].values
            
    rescorrdf_MET1_NVTCF20ps[methylc.RES_ATM_Pair] = rescfdf.mean(axis=1)
    rescorrdf_MET1_NVTCF20ps_STD[methylc.RES_ATM_Pair] = rescfdf.std(axis=1)
        
    zero_std = rescorrdf_MET1_NVTCF20ps_STD[methylc.RES_ATM_Pair][rescorrdf_MET1_NVTCF20ps_STD[methylc.RES_ATM_Pair]==0.0].index
    rescorrdf_MET1_NVTCF20
    ps_STD.loc[zero_std, methylc.RES_ATM_Pair] = 1e-6

In [ ]:
rescorrdf_MET1_NVTCF20ps[AveUBQ_NVT_Cf20ps_C13C13.columns] = AveUBQ_NVT_Cf20ps_C13C13
rescorrdf_MET1_NVTCF20ps_STD[StdUBQ_NVT_Cf20ps_C13C13.columns] = StdUBQ_NVT_Cf20ps_C13C13
rescorrdf_MET1_NVTCF20ps.to_csv('{}{}/PROD_NVT/CF20ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
rescorrdf_MET1_NVTCF20ps_STD.to_csv('{}{}/PROD_NVT/CF20ps/METHYL_C13C13_CorrFuncSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
AveUBQ_NVT_Cf02ps_C13C13.to_csv('{}{}/PROD_NVT/CF20ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
StdUBQ_NVT_Cf02ps_C13C13.to_csv('{}{}/PROD_NVT/CF20ps/METHYL_C13C13_CorrFuncAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:

AveUBQ_NVT_Cf2ps_C13H.to_csv('{}{}/PROD_NVT/CF2ps/METHYL_13CH_CorrFunctionsAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'))
StdUBQ_NVT_Cf2ps_C13H.to_csv('{}{}/PROD_NVT/CF2ps/METHYL_13CH_CorrFunctionsSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'))

## Read in Average Correlation Functions and their errors.

In [ ]:
AveUBQ_NVE_C13H = pd.read_csv('{}{}/PROD_NVE/METHYL_13CH_OptSimCorrFunctionsAverage.csv'.format(CorrFuncLoc2, 'Ubiquitin'), index_col=0)
StdUBQ_NVE_C13H = pd.read_csv('{}{}/PROD_NVE/METHYL_13CH_OptSimCorrFunctionsSTD.csv'.format(CorrFuncLoc2, 'Ubiquitin'), index_col=0)

In [ ]:
AveUBQ_NVT_Cf2ps_C13H = pd.read_csv('{}{}/PROD_NVT/CF2ps/METHYL_13CH_CorrFunctionsAverage.csv'.format(CorrFuncLoc2,'Ubiquitin'), index_col=0)
StdUBQ_NVT_Cf2ps_C13H = pd.read_csv('{}{}/PROD_NVT/CF2ps/METHYL_13CH_CorrFunctionsSTD.csv'.format(CorrFuncLoc2,'Ubiquitin'), index_col=0)

In [ ]:
lnt_steps = np.unique(np.array([5e-4*np.round(np.power(1.02,i)).astype(int) for i in range(1000)]))

In [ ]:
## Plot Correlation functions
ResCorrFig = plt.figure(1, figsize=(8,6))
for rcol in AveUBQ_NVE_C13H.columns:
    axCF = ResCorrFig.add_subplot(111)
    
    AveUBQ_NVE_C13H[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVE }$', color='blue')
    axCF.fill_between(AveUBQ_NVE_C13H.index.values,
                      AveUBQ_NVE_C13H[rcol].values-StdUBQ_NVE_C13H[rcol].values,
                      AveUBQ_NVE_C13H[rcol].values+StdUBQ_NVE_C13H[rcol].values,
                      color='b', alpha=0.5)
    
    #ResCorrDF_GB3_NVT_CF02ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    AveUBQ_NVT_Cf2ps_C13H[rcol].plot(logx=True, ax = axCF, color='green',
                                     label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
    axCF.fill_between(AveUBQ_NVT_Cf2ps_C13H.index.values,
                      AveUBQ_NVT_Cf2ps_C13H[rcol].values-StdUBQ_NVT_Cf2ps_C13H[rcol].values,
                      AveUBQ_NVT_Cf2ps_C13H[rcol].values+StdUBQ_NVT_Cf2ps_C13H[rcol].values,
                      color='green', alpha=0.5)
                                     
    #ResCorrDF_GB3_NVT_CF20ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axCF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axCF.legend(loc=3)
    axCF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axCF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axCF.tick_params(labelsize=15)
    ResCorrFig.savefig('{}Ubiquitin/Analysis/METHYL_CorrFunctions/C13HCorrelationFunctions_WithCoupling_Res{}.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    ResCorrFig.clear()
    

In [ ]:
UBQ_NVEDF[0][rcol]

In [ ]:
def fitstoDF(resnames, chi_list, pars_list, errs_list, names_list):
    ## Set Up columns indices and names for the data frame
    
    mparnames = _return_parameter_names(8)
    mtau_names = np.array(mparnames)[1::2]
    mc_names = np.array(mparnames)[::2]
    colnames = np.array(['Resname','NumExp'])
    tau_errnames = np.array([[c,"{}_err".format(c)] for c in mtau_names]).flatten()
    mc_errnames = np.array([[c, "{}_err".format(c)] for c in mc_names]).flatten()
    colnames = np.hstack([colnames,mc_errnames])
    colnames = np.hstack([colnames,tau_errnames])
    colnames = np.hstack([colnames,np.array(['Chi_Fit'])])
    FitDF = pd.DataFrame(index=np.arange(len(pars_list)), columns=colnames).fillna(0.0)
    FitDF['Resname'] = resnames
    FitDF['Chi_Fit'] = chi_list
    
    for i in range(len(pars_list)):
        npar = len(pars_list[i])
        if (npar%2)==1:
            ccut = npar-2
            tau_f, terr = pars_list[i][1:ccut+1:2], errs_list[i][1:ccut+1:2]
            tau_f = np.hstack([tau_f, pars_list[i][-1]])
            terr = np.hstack([terr, errs_list[i][-1]])
            sort_tau = np.argsort(tau_f)
            coeff, cerr= pars_list[i][0:ccut:2], errs_list[i][0:ccut:2]
            Clast = 1; Clasterr = 0.0;
            for n,m in zip(coeff, cerr):
                Clast -= n
                Clasterr += m
            
            coeff =np.hstack([coeff, np.array(Clast)])
            cerr =np.hstack([cerr, np.array(Clasterr)])
    
            tne = np.array([[c,"{}_err".format(c)] for c in mparnames[1:npar+1:2]]).flatten()
            cne = np.array([[c, "{}_err".format(c)] for c in mparnames[0:npar:2]]).flatten()
                
        else:
            tau_f, terr = pars_list[i][1::2], errs_list[i][1::2] 
            coeff, cerr= pars_list[i][0::2], errs_list[i][0::2]
            sort_tau = np.argsort(tau_f)[::-1]
            tne = np.array([[c,"{}_err".format(c)] for c in names_list[i][1::2]]).flatten()
            cne = np.array([[c, "{}_err".format(c)] for c in names_list[i][0::2]]).flatten()
    
        NumExp=np.array(len(tau_f))
        tau_err = np.array([[t,e] for t,e in zip(tau_f[sort_tau],terr[sort_tau])]).flatten()
        c_err = np.array([[c,e] for c,e in zip(coeff[sort_tau], cerr[sort_tau])]).flatten()
        namesarr = np.hstack([np.array('NumExp'),cne,tne])
        valarr = np.hstack([NumExp,c_err,tau_err])
    
        FitDF.loc[i, namesarr] = valarr
    
    FitDF['AUC_a'] = FitDF.C_a*FitDF.tau_a; FitDF['AUC_b'] = FitDF.C_b*FitDF.tau_b;
    FitDF['AUC_g'] = FitDF.C_g*FitDF.tau_g; FitDF['AUC_d'] = FitDF.C_d*FitDF.tau_d;
    FitDF['AUC_Total'] = FitDF[['AUC_a','AUC_b','AUC_g','AUC_d']].sum(axis=1)
    
    return FitDF

In [ ]:
def fitCorrF(CorrDF, dCorrDF, tau_mem, pars_l, tcorr, tscl,  tstart=1, logt=[], fixfit=False, first_neg=False):
    
    NH_Res = CorrDF.columns
    chi_list=[] ; names_list=[] ; pars_list=[] ; errs_list=[] ; ymodel_list=[]; covarMat_list = [];
    for i in CorrDF.columns:
        
        if (not first_neg):
            tstop = np.where(CorrDF.index.values==tau_mem)[0][0]
        else:
            print('Using first negative value as the final value in the fit:')
            tstop_arr = np.where(CorrDF[i].values < 0.005)[0]
            if tstop_arr.size > 0:
                tstop = tstop_arr[0]
            else:
                print('No Negative values found in the correlation')
                tstop = np.where(CorrDF.index.values==tau_mem*2)[0][0]
            print('Final Value in the Fit is at {} : {} ns'.format(tstop-1, CorrDF.index.values[tstop-1]))
        
        if len(logt) > 0:
            full_tseries = CorrDF.index.values
            logt_mask = np.array([gts for gts in logt if gts in full_tseries[tstart:tstop-1]])
            x = CorrDF.loc[logt_mask, i].index
            y = CorrDF.loc[logt_mask, i].values
        else:
            x = CorrDF.index.values[tstart:tstop-1:20]
            y = CorrDF[i].values[tstart:tstop-1:20]
            
        if len(dCorrDF) > 1:
            if len(logt) > 0:
                #logt_mask = logt[np.where(logt<tstop)]
                dy = dCorrDF.loc[logt_mask,i].values
            else:
                dy = dCorrDF[i].values[tstart:tstop-1:20]
        else:
            dy = dCorrDF
        ## if not fixfit then find find the best expstyle fit. Otherwise force the fit to nparams 
        if (not fixfit)&(len(pars_l)>1):
            print("Finding the best fit for residue {}".format(i))
            
            chi, names, pars, errs, ymodel, covarMat = findbest_Expstyle_fits2(x, y, tcorr*tscl, dy,
                                                                               par_list=pars_l, threshold_scl=thresh)
        
        elif (fixfit)&(len(pars_l)==1):
            print("Performing a fixed fit for {} exponentials for residue {}".format(int(pars_l[0]/2),i))
            ###(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005)
            chi, pars, covarMat, ymodel = do_Expstyle_fit2(pars_l[0], x, y, dy, tinput=tcorr*tscl,
                                                           tmax= tcorr*tscl, tmin=x[0])
            names = _return_parameter_names(len(pars))
            errs = np.sqrt(np.diag(covarMat))
            
        else:
            print("The list of parameters is empty. Breaking out.")
            break;
            
        chi_list.append(chi)
        names_list.append(names)
        pars_list.append(pars)
        errs_list.append(errs)
        ymodel_list.append(ymodel)
        covarMat_list.append(covarMat)
        
    FitDF = fitstoDF(NH_Res, chi_list, pars_list, errs_list, names_list)
    
    return FitDF, covarMat_list

In [ ]:
def J_direct_transform(om, consts, taus):
    
    ## Calculation for the direct spectral density 
    ndecay=len(consts) ; noms=1;###lnden(om)
    Jmat = np.zeros( (ndecay, noms ) )
    
    for i in range(ndecay):
        
        Jmat[i] = consts[i]*(taus[i]*1e-9)/(
            1 + np.power((taus[i]*1e-9)*(om),2.))
        
    return (2/5)*Jmat.sum(axis=0)

In [ ]:
def calc_NMR_MethylRelaxation(J, qcc):
    
    RDz = (3/16)*(qcc**2)*(J['2D'] + 4*J['2_2D'])
    
    RDy = (1/32)*(qcc**2)*(9*J['0'] + 15*J['2D'] + 6*J['2_2D'])
    
    return RDz, RDy

In [ ]:
def calc_NMRFit_Methyl(fit_df, magfield, exp_nmrdf=[]):
    
    """
    fit_df : dataframe containing the fitting results
    exp_nmrdf : dataframe containing the experimental NMR results
    magfield : magnetic field the experiments use (MHz)
    """
    
    ## Parameters and Physical Constants for calculation of Relaxation Rates
    H_gyro = 2*np.pi*42.57748*1e6     ## Gyromagnetic Ratio: Hydrogen ([rad]/[s][T])
    D_gyro = 2*np.pi*6.536*1e6     ## Gyromagnetic Ratio: Hydrogen ([rad]/[s][T]) 
    B0 = 2*np.pi*magfield*1e6/H_gyro   ## Field Strength = 18.8 Teslas

    ## Need 3 Frequencies: ## J[0], J[wD], J[2wD]
    Larmor2D = D_gyro*B0              ## Larmor Frequency: Deuterium ([rad]/[s]
    
    hbar = 1.0545718e-34  ; # [J] * [s] = [kg] * [m^2] * [s^-1] 
    EE = -1.602e-19
                         ## distance between N-H 
    QuadCoupleCons = 2*np.pi*(167)*1e3  ## [mHz], measured experimentally at 167 kHz (1999 Mittermaier and Kay, JACS)
    
    ## Calculate 
    Jarr = []

    for i, fit in fit_df.iterrows():
        c = fit[['C_a','C_b','C_g','C_d']].values
        t = fit[['tau_a','tau_b','tau_g','tau_d']].values
        Jdict = {'0':0, '2D':0, '2_2D':0} 
        J0 = J_direct_transform(0, c, t)
        JD = J_direct_transform(Larmor2D, c, t)
        J2D = J_direct_transform(2*Larmor2D, c, t)
        Jdict['2D'] = JD ; Jdict['2_2D'] = J2D; Jdict['0'] = J0; 
        Jarr.append(Jdict)
    
    NMRRelaxDF = pd.DataFrame(np.zeros((len(Jarr),2)), index=range(1,len(Jarr)+1), columns=['RDz', 'RDy'])
    for index in range(1,len(Jarr)+1):
        rdz, rdy = calc_NMR_MethylRelaxation(Jarr[index-1], QuadCoupleCons)
        
        NMRRelaxDF.loc[index,'RDz'] = rdz;
        NMRRelaxDF.loc[index,'RDy'] = rdy;
        
    NMRRelaxDF['Resname'] = fit_df['Resname'].values
    NMRRelaxDF['RESNUM'] = NMRRelaxDF['Resname'].astype(str).str.extract('([0-9]+)',expand=False).astype('int')
    FitRelaxDF = fit_df.merge(NMRRelaxDF, how='left', left_on='Resname', right_on='Resname').set_index(NMRRelaxDF.index)
    
    ## Calculation of RMSE between NMR values and EXP:
    if len(exp_nmrdf)>0:
        RMSE_DF = pd.DataFrame(index=NMRRelaxDF['Resname'],
                               columns = ['{}_SE'.format(nmr) for nmr in ['RDz', 'RDy']])
        for nmr in ['RDz', 'RDy']:
            RMSE_DF['{}_SE'.format(nmr)] = (NMRRelaxDF[['Resname', nmr]].set_index('Resname') - 
                                            exp_nmrdf[['Resname', nmr]].set_index('Resname'))**2
        FitRelaxDF = FitRelaxDF.merge(RMSE_DF.reset_index(), how='left', left_on='Resname', right_on='Resname')
    
    return FitRelaxDF

In [ ]:
def ScaleNMRParams(fit_nmrdf, exp_nmrdf, magfield, coupling=0.2):
    
     ## Parameters and Physical Constants for calculation of Relaxation Rates
    H_gyro = 2*np.pi*42.57748*1e6     ## Gyromagnetic Ratio: Hydrogen ([rad]/[s][T])
    D_gyro = 2*np.pi*6.536*1e6     ## Gyromagnetic Ratio: Hydrogen ([rad]/[s][T]) 
    B0 = 2*np.pi*magfield*1e6/H_gyro   ## Field Strength = 18.8 Teslas

    ## Need 3 Frequencies: ## J[0], J[wD], J[2wD]
    Larmor2D = D_gyro*B0              ## Larmor Frequency: Deuterium ([rad]/[s]
    
    hbar = 1.0545718e-34  ; # [J] * [s] = [kg] * [m^2] * [s^-1] 
    EE = -1.602e-19
                         ## distance between N-H 
    QuadCoupleCons = 2*np.pi*(167)*1e3  ## [mHz], measured experimentally at 167 kHz (1999 Mittermaier and Kay, JACS)
    
    NMRRelax_Scl = fit_nmrdf.copy()
    
    print('Scaling all correlation times by predefined scaling parameters')
        
    NMRRelax_Scl['tau_a'] = NMRRelax_Scl['tau_a'].apply(lambda tau: tau/(1.43+0.066*tau))
    NMRRelax_Scl['tau_b'] = NMRRelax_Scl['tau_b'].apply(lambda tau: tau/(1.43+0.066*tau) if tau != 5e-4 else tau) 
    
    NMRRelax_Scl['AUC_a'] = NMRRelax_Scl['tau_a']*NMRRelax_Scl['C_a']
    NMRRelax_Scl['AUC_b'] = NMRRelax_Scl['tau_b']*NMRRelax_Scl['C_b']
    NMRRelax_Scl['AUC_Total'] = NMRRelax_Scl[['AUC_a','AUC_b','AUC_g','AUC_d']].sum(axis=1)   

    for i,fit in NMRRelax_Scl.iterrows():
        c = fit[['C_a','C_b','C_g','C_d']].values
        t = fit[['tau_a','tau_b','tau_g','tau_d']].values
        Jdict = {'0':0, '2D':0, '2_2D':0} 
        J0 = J_direct_transform(0, c, t)
        JD = J_direct_transform(Larmor2D, c, t)
        J2D = J_direct_transform(2*Larmor2D, c, t)
        Jdict['2D'] = JD ; Jdict['2_2D'] = J2D; Jdict['0'] = J0; 
    
        rdz, rdy = calc_NMR_MethylRelaxation(Jdict, QuadCoupleCons)
        NMRRelax_Scl.loc[i,'RDz'] = rdz;
        NMRRelax_Scl.loc[i,'RDy'] = rdy;
        
    ## Calculation of RMSE between NMR values and EXP:
    if len(exp_nmrdf)>0:
        RMSE_DF = pd.DataFrame(index=NMRRelax_Scl['Resname'],
                               columns = ['{}_SE'.format(nmr) for nmr in ['RDz', 'RDy']])
        for nmr in ['RDz', 'RDy']:
            RMSE_DF['{}_SE'.format(nmr)] = (NMRRelax_Scl[['Resname', nmr]].set_index('Resname') - 
                                            exp_nmrdf[['Resname', nmr]].set_index('Resname'))**2
        NMRRelax_Scl = NMRRelax_Scl.merge(RMSE_DF.reset_index(), how='left', left_on='Resname', right_on='Resname')

    return NMRRelax_Scl
       

In [ ]:
tau_mem=10.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_2exp_TS1, covarMat_lists = fitCorrF(AveUBQ_NVE_C13H, StdUBQ_NVE_C13H,
                                                        tau_mem, [4],
                                                         tstart=1, tcorr=4.11, tscl=2.0,
                                                         fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=10.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(AveUBQ_NVE_C13H, StdUBQ_NVE_C13H,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=2.0,
                                                            logt=lnt_steps,
                                                            fixfit=FixFit, first_neg=False)

In [ ]:
FitDF_UBQ_NVE_FixTM_2exp_TS1.plot(y='tau_b', yerr='tau_b_err',logy=True)
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(y='tau_b', yerr='tau_b_err', logy=True, ax=plt.gca())
plt.gca().set_ylim(5e-4,1.1)

In [ ]:
FitDF_UBQ_NVE_FixTM_2exp_TS1.plot(y='C_b', yerr='C_b_err')
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(y='C_b', yerr='C_b_err', ax=plt.gca())


In [ ]:
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT['tau_a'].mean()

In [ ]:
tau_mem=11.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_3exp_TS1, covarMat_lists = fitCorrF(AveUBQ_NVE_C13H,
                                                    StdUBQ_NVE_C13H, tau_mem, [6], 1, fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=12.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_4exp, covarMat_lists = fitCorrF(AveUBQ_NVE_C13H,
                                     StdUBQ_NVE_C13H, tau_mem, [8], 0, fixfit=FixFit, first_neg=False)

In [ ]:
FitDF_GB3_NVE_FixTM_t00.to_csv('{}{}/PROD_NVE/METHYL_C13H_FitDF_Fix3Exp_tm11ns.csv()')

In [ ]:
tau_mem=16.0;
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1, covarMat_lists = fitCorrF(AveUBQ_NVT_Cf2ps_C13H, StdUBQ_NVT_Cf2ps_C13H,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=3.53, tscl=2.0,
                                                            logt=[],
                                                            fixfit=True, first_neg=False)

In [ ]:
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(AveUBQ_NVT_Cf2ps_C13H, StdUBQ_NVT_Cf2ps_C13H,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=3.53, tscl=2.0,
                                                            logt=lnt_steps,
                                                            fixfit=True, first_neg=False)

In [ ]:
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1, covarMat_lists = fitCorrF(AveUBQ_NVT_Cf2ps_C13H, StdUBQ_NVT_Cf2ps_C13H, 
                                                        11.0, [6], 1, fixfit=FixFit, first_neg=False)

### C13-C13 Methyl Correlation Function fitting

In [ ]:
tau_mem=10.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_2exp_TS1, covarMat_lists = fitCorrF(rescorrdf_MET1_ONVE, rescorrdf_MET1_ONVE_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=2.0,
                                                            logt=[],
                                                            fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=10.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(rescorrdf_MET1_ONVE, rescorrdf_MET1_ONVE_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=2.0,
                                                            logt=lnt_steps,
                                                            fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=12.0;
FitDFC13C13_UBQ_NVT_02ps_FixTM_2exp_TS1, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF02ps, rescorrdf_MET1_NVTCF02ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=2.0,
                                                            logt=[],
                                                            fixfit=True, first_neg=False)

In [ ]:
FitDFC13C13_UBQ_NVT_02ps_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF02ps, rescorrdf_MET1_NVTCF02ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=2.0,
                                                            logt=lnt_steps,
                                                            fixfit=True, first_neg=False)

In [ ]:
tau_mem=16.0;
FitDFC13C13_UBQ_NVT_2ps_FixTM_2exp_TS1, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF2ps, rescorrdf_MET1_NVTCF2ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=3.0,
                                                            logt=[],
                                                            fixfit=True, first_neg=False)

In [ ]:
tau_mem=16.0;
FitDFC13C13_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF2ps, rescorrdf_MET1_NVTCF2ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=3.0,
                                                            logt=lnt_steps,
                                                            fixfit=True, first_neg=False)

In [ ]:
tau_mem=50.0;
FitDFC13C13_UBQ_NVT_20ps_FixTM_2exp_TS1, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF20ps, rescorrdf_MET1_NVTCF20ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=15.0,
                                                            logt=[],
                                                            fixfit=True, first_neg=False)

In [ ]:

FitDFC13C13_UBQ_NVT_20ps_FixTM_2exp_TS1_LnT, covarMat_lists = fitCorrF(rescorrdf_MET1_NVTCF20ps, rescorrdf_MET1_NVTCF20ps_STD,
                                                            tau_mem, [4],
                                                            tstart=1, tcorr=4.11, tscl=15.0,
                                                            logt=lnt_steps,
                                                            fixfit=True, first_neg=False)

In [ ]:
FitDFC13C13_UBQ_NVT_02ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_a', yerr='C_a_err')
FitDFC13C13_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_a',yerr='C_a_err', ax=plt.gca())
FitDFC13C13_UBQ_NVT_20ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_a',yerr='C_a_err', ax=plt.gca())

In [ ]:
NMRFitDF_C13C13_OptSimNVE_TS1_LnT = calc_NMRFit_Methyl(FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT, 600, UBQ_MethylRelax)
#NMRFitDF_C13C13_NVT_CF02ps_TS1_LnT = calc_NMRFit_Methyl(FitDFC13C13_UBQ_NVT_02ps_FixTM_2exp_TS1_LnT,
#                                                        600, UBQ_MethylRelax)
#NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT = calc_NMRFit_Methyl(FitDFC13C13_UBQ_NVT_2ps_FixTM_2exp_TS1, 600, UBQ_MethylRelax)
#NMRFitDF_C13C13_NVT_CF20ps_TS1_LnT = calc_NMRFit_Methyl(FitDFC13C13_UBQ_NVT_20ps_FixTM_2exp_TS1, 600, UBQ_MethylRelax)

In [ ]:
EXPMethylS2 = pd.read_csv('{}{}/Ubiquitin_MethylSC_S2Axis_Captured.csv'.format(CorrFuncLoc2, 'Ubiquitin'), 
                          header=None, names=['SC_Methyl',r'$S^{2}_{Exp}$'])

In [ ]:
EXPMethylS2 = pd.read_csv('{}{}/Ubiquitin_MethylSC_S2Axis_Captured.csv'.format(CorrFuncLoc2, 'Ubiquitin'), 
                          header=None, names=['SC_Methyl',r'$S^{2}_{Exp}$'])
EXPMethylS2['RESNAME'] = EXPMethylS2['SC_Methyl'].apply(lambda scm : scm[0] )
EXPMethylS2['RESID'] = EXPMethylS2['SC_Methyl'].str.findall('([0-9]+)').apply(lambda resi: resi[0])
EXPMethylS2['METHYL_ATOMCode'] = EXPMethylS2['SC_Methyl'].str.findall('([A-Za-z]+)').apply(lambda atm: 'C{}'.format(atm[-1]).upper())
EXPMethylS2['METHYL_ATOMNum'] = EXPMethylS2['SC_Methyl'].str.findall('([A-Za-z]+)([0-9]+)').apply(lambda scid : scid[-1][1] if len(scid) > 1 else '')
EXPMethylS2['SCPair_key'] = (EXPMethylS2[['RESNAME', 'RESID', 'METHYL_ATOMCode', 'METHYL_ATOMNum']]
                             .apply(lambda scpair: '{}{}-{}{}'.format(scpair.RESNAME,scpair.RESID,scpair.METHYL_ATOMCode,scpair.METHYL_ATOMNum),axis=1))

In [ ]:
Three2OneAA = {'Ala':'A', 'ALA':'A', 'Arg':'R', 'ARG':'R', 'ASP':'D', 'Asp':'D', 'Asn':'N', 'ASN':'N',
               'Cys':'C', 'CYS':'C', 'Glu':'E', 'GLU':'E', 'Gln':'Q', 'GLN':'Q', 'Gly':'G', 'GLY':'G',
               'His':'H', 'HIS':'H', 'HID':'H', 'Hid':'H', 'HIE':'H', 'Hie':'H', 'ILE':'I', 'Ile':'I',
               'LEU':'L', 'Leu':'L', 'LYS':'K', 'Lys':'K', 'MET':'M', 'Met':'M', 'PHE':'F', 'Phe':'F',
               'Pro':'P', 'PRO':'P', 'Ser':'S', 'SER':'S', 'THR':'T', 'Thr':'T', 'Trp':'W', 'TRP':'W',
               'Tyr':'Y', 'TYR':'Y', 'Val':'V', 'VAL':'V'}

In [ ]:
NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT['SC_PairKey'] = NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT['Resname'].apply(lambda rr : '{}{}-{}'.format(Three2OneAA[rr[:3]],rr.split('-')[0][3:],rr.split('-')[1]))
NMRFitDF_C13C13_NVT_CF02ps_TS1_LnT['SC_PairKey'] = NMRFitDF_C13C13_NVT_CF02ps_TS1_LnT['Resname'].apply(lambda rr : '{}{}-{}'.format(Three2OneAA[rr[:3]],rr.split('-')[0][3:],rr.split('-')[1]))
NMRFitDF_C13C13_NVT_CF20ps_TS1_LnT['SC_PairKey'] = NMRFitDF_C13C13_NVT_CF20ps_TS1_LnT['Resname'].apply(lambda rr : '{}{}-{}'.format(Three2OneAA[rr[:3]],rr.split('-')[0][3:],rr.split('-')[1]))

NMRFitDF_C13C13_OptSimNVE_TS1_LnT['SC_PairKey'] = NMRFitDF_C13C13_OptSimNVE_TS1_LnT['Resname'].apply(lambda rr : '{}{}-{}'.format(Three2OneAA[rr[:3]],rr.split('-')[0][3:],rr.split('-')[1]))

In [ ]:
NMRFitDF_C13C13_OptSimNVE_TS1_LnT.to_csv('{}{}/PROD_NVE/NMRFitDF_C13C13_FixTM10ns_OptSimNVE_TS1_LnT.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
#NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT.to_csv('{}{}/PROD_NVT/CF2ps/NMRFitDF_C13C13_FixTM16ns_NVT_CF2ps_TS1_LnT.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
#NMRFitDF_C13C13_NVT_CF02ps_TS1_LnT.to_csv('{}{}/PROD_NVT/CF0-2ps/NMRFitDF_C13C13_FixTM13ns_NVT_CF02ps_TS1_LnT.csv'.format(CorrFuncLoc2,'Ubiquitin'))
#NMRFitDF_C13C13_NVT_CF20ps_TS1_LnT.to_csv('{}{}/PROD_NVT/CF20ps/NMRFitDF_C13C13_FixTM50ns_NVT_CF20ps_TS1_LnT.csv'.format(CorrFuncLoc2, 'Ubiquitin'))


In [ ]:
NMRFitDF_C13C13_OptSimNVE_TS1_LnT

In [ ]:
NMRFitDF_C13C13_NVT_CF2ps_4Plot = NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT.set_index('SC_PairKey').loc[EXPMethylS2['SCPair_key'].values].reset_index()
NMRFitDF_C13C13_NVT_CF02ps_4Plot = NMRFitDF_C13C13_NVT_CF02ps_TS1_LnT.set_index('SC_PairKey').loc[EXPMethylS2['SCPair_key'].values].reset_index()
NMRFitDF_C13C13_NVT_CF20ps_4Plot = NMRFitDF_C13C13_NVT_CF2ps_TS1_LnT.set_index('SC_PairKey').loc[EXPMethylS2['SCPair_key'].values].reset_index()

NMRFitDF_C13C13_OptSimNVE_4Plot = NMRFitDF_C13C13_OptSimNVE_TS1_LnT.set_index('SC_PairKey').loc[EXPMethylS2['SCPair_key'].values].reset_index()

In [ ]:
UBQ_OPTNVE_C13S2 = pd.read_csv('{}{}/PROD_NVE/UBQ_SCMethyl_S2Axis_OptSimNVE.csv'.format(CorrFuncLoc2, 'Ubiquitin'), index_col=0)
UBQ_C13S2_CF2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/UBQ_SCMethyl_S2Axis_NVT_CF2ps.csv'.format(CorrFuncLoc2, 'Ubiquitin'), index_col=0)

In [ ]:
UBQ_OptNVEDF_C13C13[0].iloc[:,30].plot(logx=True)
ymodel_NVEdf = func_exp_decay4(UBQ_OptNVEDF_C13C13[0].index.values, 
                               *FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.iloc[30][['C_a','tau_a','C_b','tau_b']].values)
plt.semilogx(UBQ_OptNVEDF_C13C13[0].index.values, ymodel_NVEdf)

In [ ]:
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT['tau_b']

In [ ]:
EXPMethylS2.plot(y=r'$S^{2}_{Exp}$',label='Experiment',color='k')
UBQ_OPTNVE_C13S2.loc[EXPMethylS2['SCPair_key'].values].mean(axis=1).plot(label=r'$\mathbf{NVE \ \ / \ \ P2}$', 
                                                                         ax=plt.gca(), color='blue')
NMRFitDF_C13C13_OptSimNVE_4Plot.plot(y='C_a', ax=plt.gca(), label=r'$\mathbf{NVE \ \ / \ \ A_{1}}$', color='aquamarine')
plt.gca().legend(frameon=False)
plt.savefig('{}{}/SCMethyl_C13C13_OrderParameters_P2vsAl.png'.format(CorrFuncLoc2,'Ubiquitin'),
            dpi=600, bbox_inches='tight')

In [ ]:
figC13S2UBQ = plt.figure(3289, figsize=(8, 6))
axC13S2ubq = figC13S2UBQ.add_subplot(111)

EXPMethylS2[['SCPair_key', r'$S^{2}_{Exp}$']].plot( y=r'$S^{2}_{Exp}$', color='k', style='-..',
                                          label=r'$\mathbf{Experiment}$',
                                          ax=axC13S2ubq, linewidth=2.5)

NMRFitDF_C13C13_OptSimNVE_4Plot.plot(y='C_a', style='-x', color='blue', markeredgecolor='k',
                                     label = r'$\mathbf{NVE}$', ax = axC13S2ubq,
                                     linewidth=2.5, markersize=6.5, alpha=0.75)

NMRFitDF_C13C13_NVT_CF02ps_4Plot.plot(y='C_a', style='-s',color='orange', markeredgecolor='k',
                                      label = r'$\mathbf{NVT: \zeta = 0.2 \ \ ps^{-1}}$',
                                      ax = axC13S2ubq, linewidth=2.5,
                                      markersize=6.5, alpha=0.75)

NMRFitDF_C13C13_NVT_CF2ps_4Plot.plot(y='C_a', style='-o', color='green', markeredgecolor='k',
                                     label = r'$\mathbf{NVT: \zeta = 2 \ \ ps^{-1}}$',
                                     ax = axC13S2ubq, linewidth=2.5,
                                     markersize=6.5, alpha=0.75)

NMRFitDF_C13C13_NVT_CF20ps_4Plot.plot(y='C_a', style='-d', color='red', markeredgecolor='k',
                                      label = r'$\mathbf{NVT: \zeta = 20 \ \ ps^{-1}}$', 
                                      ax = axC13S2ubq, linewidth=2.5,
                                      markersize=6.5, alpha=0.75)


axC13S2ubq.set_ylabel(r'$\mathbf{S^2_{axis}}$', fontsize=18)
axC13S2ubq.set_xlabel(r'Methyl Number', fontsize=18, weight='bold')
axC13S2ubq.tick_params(labelsize=17)
axC13S2ubq.set_ylim(0.0,1.25)
#axC13S2ubq.set_xticks(np.arange(0,80,5), minor=True)
axC13S2ubq.set_yticks(np.arange(0.0, 1.2, 0.2))
axC13S2ubq.set_yticks(np.arange(0, 1.1, 0.1), minor=True)
## Split the legend
axC13hndls, axC13lbls = axC13S2ubq.get_legend_handles_labels()
nve_legend = axC13S2ubq.legend(axC13hndls[:2], axC13lbls[:2],
                               frameon=False, loc='upper left', prop={'size':14})
nvelgax = axC13S2ubq.add_artist(nve_legend)

axC13S2ubq.legend(axC13hndls[2:], axC13lbls[2:], frameon=False, loc='upper right', prop={'size':14})

figC13S2UBQ.savefig('{}{}/Analysis/FigureS3_Ubiquitin_SCMehtylOrderParameters_AllSims_CompareExp_C13C13A1.png'.format(CorrFuncLoc2, 'Ubiquitin'),
                    dpi=600, bbox_inches='tight')

In [ ]:
axca = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='C_a', color='blue')
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname',y='C_a',color='green',ax=axca)

axcb = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='C_b', color='blue')
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname', y='C_b', color='green', ax=axcb)

axcg = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='C_g', color='blue')
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname', y='C_g', color='green', ax=axcg)

In [ ]:
axta = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='tau_a', color='blue', logy=True)
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname', y='tau_a',color='green', ax=axta, logy=True)

axtb = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='tau_b', color='blue', logy=True)
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname', y='tau_b', color='green', ax=axtb, logy=True)

axtg = FitDF_UBQ_NVE_FixTM_3exp_TS1.plot(x='Resname', y='tau_g', color='blue', logy=True)
FitDF_UBQ_NVT_2ps_FixTM_3exp_TS1.plot(x='Resname', y='tau_g', color='green', ax=axtg ,logy=True)

In [ ]:
FitDF_UBQ_NVE_FixTM_2exp['tau_a']

In [ ]:
axca = FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_a', color='blue')
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_a',color='green', ax=axca)

axcb = FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_b', color='blue')
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='C_b', color='green', ax=axcb)

In [ ]:
axta = FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(x='Resname', y='tau_a', color='blue', logy=True)
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='tau_a',color='green', ax=axta, logy=True)

axtb = FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.plot(x='Resname', y='tau_b', color='blue', logy=True)
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.plot(x='Resname', y='tau_b', color='green', ax=axtb, logy=True)

In [ ]:
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1

In [ ]:
def _plot_Figure7(fitdf_nve, fitdf_nvt, tcorr_nve, tcorr_nvt, fsize=(10,18), ylim_t1=(1,8)):
    
    fig_f3, axf3 = plt.subplots(3, 1,  figsize=fsize, sharex=True)
    fig_f3.subplots_adjust(hspace=0.05)
    
    ## Plot the Amplitudes to S1
    fitdf_nve[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
                                                                ax=axf3[0], label=r'$\mathbf{A_1-NVE}$',
                                                         linewidth=2, marker='o', linestyle='-')


    fitdf_nvt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='green',
                                                               ax=axf3[0],
                                                               label=r'$\mathbf{A_1 - NVT : \zeta = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d', linestyle='-')
    
    
    fitdf_nve[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
                                                ax=axf3[0],  label=r'$\mathbf{A_2 - NVE}$',
                                                linewidth=2, marker='o', linestyle='-.')


    fitdf_nvt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='green',
                                                ax=axf3[0],
                                                label=r'$\mathbf{A_2 - NVT : \zeta = 2 \ \ ps^{-1}}$',
                                                linewidth=2, marker='d', linestyle='-.')
    
    axf3[0].set_ylim(0, 1)
    axf3[0].set_ylabel(r'$\mathbf{A_{1} \ \ and \ \ A_{2}}$', fontsize=15, weight='bold')
    axf3[0].legend(loc=7, frameon=False, bbox_to_anchor=(1.0,0.375))
    
    
    ## Plot the Tau_1 to S2
    fitdf_nve[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
                                                                    ax=axf3[1], label=r'$\mathbf{NVE}$',
                                                             linewidth=2, marker='o')
    axf3[1].axhline(tcorr_nve, xmin=0, xmax=fitdf_nve.shape[0], 
                    linewidth=2, linestyle='--', color='blue')
    
    fitdf_nvt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='green',
                                                                   ax=axf3[1],
                                                                   label=r'$\mathbf{ NVT : \zeta = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
    axf3[1].axhline(tcorr_nvt, xmin=0, xmax=fitdf_nve.shape[0],
                    linewidth=2, linestyle='--', color='green')
    
    if (ylim_t1[1]-ylim_t1[0]) > 10:
        axf3[1].tick_params(labelsize=15, which='major')
    else: 
        axf3[1].tick_params(labelsize=15, which='both')
        axf3[1].set_yticks(np.linspace(np.ceil(ylim_t1[0]).astype(int), ylim_t1[1],
                                        ylim_t1[1] - np.ceil(ylim_t1[0]).astype(int) + 1)[1:-1], minor=True)
        axf3[1].set_yticklabels(np.linspace(np.ceil(ylim_t1[0]).astype(int), ylim_t1[1],
                                            ylim_t1[1] - np.ceil(ylim_t1[0]).astype(int) + 1)[1:-1], minor=True)
        axf3[1].yaxis.set_minor_formatter(mticker.ScalarFormatter())
        
    axf3[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:.0f}'))
    
    axf3[1].set_ylim(ylim_t1[0],ylim_t1[1])
    axf3[1].set_ylabel(r'$\mathbf{\tau_{1} \ \ (ns)}$', fontsize=15, weight='bold')
    axf3[1].legend(loc='best', frameon=False)
    
    ## Plot Tau_2 to S3 
    fitdf_nve[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
                                                                    ax=axf3[2], label=r'$\mathbf{NVE}$',
                                                             logy=True, linewidth=2, marker='o')
    
    fitdf_nvt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='green',
                                                                   ax=axf3[2],
                                                                   label=r'$\mathbf{ NVT : \zeta = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
    axf3[2].set_xlabel('Residue', fontsize=15, weight='bold')
    axf3[2].set_ylabel(r'$\mathbf{\tau_{2} \ \ (ns)}$', fontsize=15, weight='bold')
    axf3[2].set_ylim(3e-4, 2e0)
    axf3[2].set_xlim(-2, fitdf_nve.shape[0])
    axf3[2].legend(loc='best', frameon=False)
    
    for axfit in axf3:
        axfit.set_xlim(0, fitdf_nve.shape[0]+1)
        axfit.set_xticks(np.arange(0, fitdf_nve.shape[0], 10))
        axfit.set_xticks(np.arange(0, fitdf_nve.shape[0], 5), minor=True)
        axfit.tick_params(labelsize=15)
        
    axf3[0].set_xticklabels([])
    axf3[1].set_xticklabels([])
    axf3[2].set_xticklabels(np.arange(0,fitdf_nve.shape[0],10))
    axf3[2].set_xlim(-1, fitdf_nve.shape[0])
    
    fig_f3.canvas.draw()
    ## Align Ylabels
    f3s1_ylbl_pos = axf3[0].yaxis.label.get_position()
    f3s2_ylbl_pos = axf3[1].yaxis.label.get_position()
    f3s3_ylbl_pos = axf3[2].yaxis.label.get_position()
    print(f3s1_ylbl_pos, f3s2_ylbl_pos, f3s3_ylbl_pos)
    axf3[1].set_ylabel(r'$\mathbf{\tau_{1} \ \ (ns)}$', labelpad = (f3s2_ylbl_pos[0] - f3s3_ylbl_pos[0] + 4.0),
                       fontsize=15, weight='bold')
    axf3[0].set_ylabel(r'$\mathbf{A_{1} \ \ or \ \ A_{2}}$', labelpad = (f3s1_ylbl_pos[0] - f3s3_ylbl_pos[0] + 4.0),
                       fontsize=15, weight='bold')

    
    return fig_f3

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp_TS1_LnT = pd.read_csv('{}{}/PROD_NVE/NMRMethylRelaxation_FitDF_UBQ_OPTSim_2Exp_TS1_LnT.csv'.format(CorrFuncLoc2,'Ubiquitin'),
                                          index_col=0)
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT = pd.read_csv('{}{}/PROD_NVT/CF2ps/NMRMethylRelaxation_FitDF_UBQ_2Exp_TS1_LnT.csv'.format(CorrFuncLoc2,'Ubiquitin'),
                                             index_col=0)

In [ ]:
figMethylFits_TS0 = _plot_Figure7(NMRFitDF_UBQ_NVE_FixTM_2exp_TS0.dropna(), NMRFitDF_UBQ_NVT_2ps_FixTM_2exp_TS0.dropna(), 4.33, 8.77, fsize=(8,10), ylim_t1=(0.5, 11))
figMethylFits_TS0.savefig('{}{}/Analysis/MethylFitParameters_UBQ_newBounds_TS0.png'.format(CorrFuncLoc2,'Ubiquitin'),
                      bbox_inches='tight', dpi=600)

In [ ]:
figMethylFits = _plot_Figure7(NMRFitDF_UBQ_NVE_FixTM_2exp_TS1_LnT.dropna(), NMRFitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT.dropna(), 4.33, 8.77, fsize=(8,10), ylim_t1=(0.5, 12))
figMethylFits.savefig('{}{}/Analysis/MethylFitParameters_UBQ_TS1_LnT.png'.format(CorrFuncLoc2,'Ubiquitin'),
                      bbox_inches='tight', dpi=600)

In [ ]:
UBQ_ExpRDy  = pd.read_excel('{}{}/ExpMethylRelaxation_RDx_RDy.xlsx'.format(CorrFuncLoc2, 'Ubiquitin'), sheet_name='RDy')
UBQ_ExpRDz  = pd.read_excel('{}{}/ExpMethylRelaxation_RDx_RDy.xlsx'.format(CorrFuncLoc2, 'Ubiquitin'), sheet_name='RDz')

In [ ]:
UBQ_MethylRelax = UBQ_ExpRDy[['Methyl','RNMR']].copy().rename(columns={'Methyl':'Resname','RNMR':'RDy'})
UBQ_MethylRelax['RDz'] = UBQ_ExpRDz['RNMR']

In [ ]:
FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT['Resname'] = METHYLCarbon_df['RES_ATM_Pair']
FitDF_UBQ_NVE_FixTM_2exp_TS1['Resname'] = METHYLCarbon_df['RES_ATM_Pair']
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT['Resname'] = METHYLCarbon_df['RES_ATM_Pair']
FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1['Resname'] = METHYLCarbon_df['RES_ATM_Pair']

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp = calc_NMRFit_Methyl(FitDF_UBQ_NVE_FixTM_2exp_TS1_LnT, 600, UBQ_MethylRelax)
NMRFitDF_UBQ_NVE_FixTM_2exp_TS1 = calc_NMRFit_Methyl(FitDF_UBQ_NVE_FixTM_2exp_TS1, 600, UBQ_MethylRelax)
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp = calc_NMRFit_Methyl(FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1_LnT, 600, UBQ_MethylRelax)
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp_TS1 = calc_NMRFit_Methyl(FitDF_UBQ_NVT_2ps_FixTM_2exp_TS1, 600, UBQ_MethylRelax)

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.to_csv('{}{}/PROD_NVE/NMRMethylRelaxation_FitDF_UBQ_OPTSim_2Exp_TS1_LnT.csv'.format(CorrFuncLoc2,'Ubiquitin'))
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp.to_csv('{}{}/PROD_NVT/CF2ps/NMRMethylRelaxation_FitDF_UBQ_2Exp_TS1_LnT.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp_TS0['RDy']/NMRFitDF_UBQ_NVE_FixTM_2exp['RDy']

In [ ]:
NMRFitDF_UBQ_sNVT = ScaleNMRParams(NMRFitDF_UBQ_NVT_2ps_FixTM_2exp, UBQ_MethylRelax, 600)

In [ ]:
NMRFitDF_UBQ_sNVT_TS1 = ScaleNMRParams(NMRFitDF_UBQ_NVT_2ps_FixTM_2exp_TS1, UBQ_MethylRelax, 600)

In [ ]:
NMRFitDF_UBQ_sNVT.dropna()['tau_a'].plot(color='g')
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp.dropna()['tau_a'].plot()

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().iloc[16]

In [ ]:
figMCNMR, axmcnmr = plt.subplots(2, 1, sharex=True, figsize=(8,12) ,num=30291)
figMCNMR.subplots_adjust(hspace=0.05)

axRDy = axmcnmr[0]
UBQ_MethylRelax.dropna().reset_index(drop=True).plot( y='RDy',color = 'k',
                                                     ax=axRDy, label=r'$\mathbf{Exp}$',
                                                     linestyle='--', linewidth=2, marker='d')

NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().reset_index(drop=True).plot(y='RDy', color='blue',
                                                                 ax=axRDy, label=r'$\mathbf{NVE}$',
                                                                 linestyle='-', linewidth=2, marker='d')

NMRFitDF_UBQ_sNVT.dropna().reset_index(drop=True).plot(y='RDy', color='green',
                                                       ax=axRDy,  label=r'$\mathbf{sNVT:  \gamma = 2 \ \ ps^{-1}}$',
                                                       linestyle='-', linewidth=2, marker='o')

axRDy.set_ylabel(r'$\mathbf{R(D_{y}) \ \ (s^{-1}) }$', fontsize=15)
axRDy.tick_params(labelsize=15)
axRDy.legend(frameon=False, prop={'size':15})

axRDz = axmcnmr[1]
UBQ_MethylRelax.dropna().reset_index(drop=True).plot(y='RDz',  color='k',
                                                     ax=axRDz,  label=r'$\mathbf{Exp}$', 
                                                     linestyle='--', linewidth=2, marker='d' )
NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().reset_index(drop=True).plot(y='RDz', color='blue',
                                                                 ax=axRDz, label=r'$\mathbf{NVE}$', 
                                                                 linestyle='-', linewidth=2, marker='d')
NMRFitDF_UBQ_sNVT.dropna().reset_index(drop=True).plot(y='RDz', ax=axRDz, color='green',
                                                       label=r'$\mathbf{sNVT:  \gamma = 2 \ \ ps^{-1}}$',
                                                       linestyle='-', linewidth=2, marker='o')

axRDz.set_ylabel(r'$\mathbf{R(D_{z}) \ \ (s^{-1}) }$', fontsize=15)
axRDz.legend(frameon=False, prop={'size':15})
axRDz.tick_params(labelsize=15)

figMCNMR.savefig('{}{}/Analysis/MethylRelaxation_ScaledCompare_2exp_FixTM_TS1_LnT.png'.format(CorrFuncLoc2, 'Ubiquitin'),
                 dpi=600, bbox_inches='tight')

In [ ]:
### Dependence on RDy RDz Single Exponential 

In [ ]:
lll = [0.9, 3.05, 1.33]

In [ ]:
tauc_series

In [ ]:
time_series = np.arange(5e-4, 100, 0.005)
tauc_series = np.geomspace(1e-3, 1e1, 101)
Test_FitDF = pd.DataFrame(index = np.arange(0, tauc_series.shape[0],1),
                          columns=['Resname','C_a','C_b','C_g','C_d','tau_a','tau_b','tau_g','tau_d']).fillna(0.0)
Test_FitDF['Resname'] = np.arange(0, tauc_series.shape[0], 1)+1
Test_FitDF['C_a'] = 0.0472621
Test_FitDF['C_b'] = 0.623701
Test_FitDF['tau_a'] = 3.057
Test_FitDF['tau_b'] = tauc_series
NMRFitTest = calc_NMRFit_Methyl(Test_FitDF, 600)

In [ ]:
Test_FitDF_alter= pd.DataFrame(index = np.arange(0,tauc_series.shape[0],1),
                          columns=['Resname','C_a','C_b','C_g','C_d','tau_a','tau_b','tau_g','tau_d']).fillna(0.0)
Test_FitDF_alter['Resname'] = np.arange(0, tauc_series.shape[0], 1)+1
Test_FitDF_alter['C_a'] = 0.005
Test_FitDF_alter['C_b'] = 0.9
Test_FitDF_alter['tau_a'] = 4.11
Test_FitDF_alter['tau_b'] = tauc_series
NMRFitTest_alter = calc_NMRFit_Methyl(Test_FitDF_alter, 600)

In [ ]:
NMRFitTest_alter['RDy']/NMRFitTest['RDy']

In [ ]:
figSCtest, axsct = plt.subplots(1, 1, figsize=(8,6))
NMRFitTest.plot(x='tau_b', y='RDz', logx=True,logy=True, ax=axsct)
NMRFitTest.plot(x='tau_b', y='RDy', logx=True, logy=True,  ax=axsct)
#NMRFitTest_alter.plot(x='tau_b', y='RDz', ax=axsct, linestyle='--')
#NMRFitTest_alter.plot(x='tau_b', y='RDy', ax=axsct, linestyle='--')
axsct.grid(True)
axsct.set_ylabel('R(Dy) or R(Dz)', weight='bold', fontsize=15)
axsct.set_xlabel(r'$\mathbf{\tau_{c} \ \ (ns)}$', fontsize=15)
axsct.tick_params(labelsize=15)
axsct.set_xlim(1e-3, 1)
axsct.axvline(0.0714859, color='k', linestyle='--')
axsct.axhline(NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().iloc[16]['RDy'], color='k', linestyle='--')
figSCtest.savefig('{}TestingSCMethlyRelaxation_Dependence_BiExponential_LEU43-CD2.png'.format(CorrFuncLoc2),dpi=600,bbox_inches='tight')

In [ ]:
axRDz = NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().plot(x='Resname', y='RDy', color='blue')
NMRFitDF_UBQ_NVT_2ps_FixTM_2exp.dropna().plot(x='Resname',y='RDz', ax=axRDz, color='green')
UBQ_MethylRelax.dropna().plot(x='Resname', y='RDy', ax=axRDz)

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().iloc[4][['C_a','tau_a','C_b','tau_b']].values.astype(float)

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.dropna()['C_a'] - NMRFitDF_UBQ_NVT_2ps_FixTM_2exp.dropna()['C_a']

In [ ]:
model_nve = func_exp_decay4(AveUBQ_NVE_C13H.index.values, 
                            *NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().iloc[29][['C_a','tau_a','C_b','tau_b']].values.astype(float))
model_nvt = func_exp_decay4(AveUBQ_NVE_C13H.index.values, 
                            *NMRFitDF_UBQ_NVT_2ps_FixTM_2exp.dropna().iloc[29][['C_a','tau_a','C_b','tau_b']].values.astype(float))


In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp[['Resname','Chi_Fit']].sort_values('Chi_Fit')

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.sort_values('C_b')['C_b']

In [ ]:
NMRFitDF_UBQ_NVE_FixTM_2exp.dropna().iloc[29]

In [ ]:
AveUBQ_NVE_C13H.columns

In [ ]:
AveUBQ_NVE_C13H.loc[:, 'LEU70-CD1'].plot(logx=True)
plt.semilogx(AveUBQ_NVE_C13H.index.values, model_nve)

In [ ]:
plt.semilogx(np.fft.rfftfreq(AveUBQ_NVE_C13H.index.values.shape[0], 5e-4), np.fft.rfft(model_nve))
plt.semilogx(np.fft.rfftfreq(AveUBQ_NVE_C13H.index.values.shape[0], 5e-4), np.fft.rfft(model_nvt))

In [ ]:
tarfname = '{}{}/METHYL_C13HCorrFuncData.tar.gz'.format(CorrFuncLoc2, 'Ubiquitin')
with tarfile.open(tarfname, 'r:gz', encoding='utf-8') as corrtar:
    
    tarmemberloc = './PROD_NVE/Analysis/{}/METHYLCorrFunc/13C{}_H1{}CorrelationFunctions.dat'.format('Run1',
                                                                                                     METHYLCarbon_df.loc[0,'ATMID'],
                                                                                                     METHYLCarbon_df.loc[0,'RESID'])
    cffile_extract = corrtar.extractfile(tarmemberloc).read()
    CorrFuncDF = pd.read_csv(io.BytesIO(cffile_extract), delim_whitespace=True, encoding='utf8')
        

In [ ]:
METHYCorrMINDX = pd.MultiIndex.from_product([ METHYLCarbonName.astype(str).values,['H1','H2','H3']])

In [ ]:
pd.DataFrame(columns=METHYCorrMINDX,index = CorrFuncDF.index)